# Dynamics

Reproduces the dynamics results: **Fig. 2b–2e** and **Figs. S3, S4**.

Each trained RNN is driven under two input regimes — the DMS-D task, and unstructured
noise — and its hidden-state trajectory is summarised by PCA. We then ask how much of
the empirical fMRI variance those five-dimensional subspaces capture, relative to a
null of random subspaces of the same dimensionality.

**Requires:** trained models for rows 0–3 of the params CSV, **and** the HCP fMRI files
(see [`data_private/README.md`](../data_private/README.md)). Point `fmri_dir` in
`paths.yaml` at wherever they live.

Analysis code lives in [`src/dynamics.py`](../src/dynamics.py),
[`src/pca_utils.py`](../src/pca_utils.py) and [`src/null_utils.py`](../src/null_utils.py).

> Evaluation is **seeded** (`dynamics.EVAL_SEED`). The task battery and the noise drive
> are random draws, so without a seed every PC, variance-explained value and z-score
> shifts between executions.

In [ ]:
import os
import warnings

import gymnasium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm

import src.dynamics as dyn
import src.null_utils as nu
import src.pca_utils as pca_utils
import src.performance as perf
import src.plotting as plotting
import src.spatial_null as spatial_null
import src.utils as utils
from src.config import ensure_dir, get_paths
from src.fmri_io import load_fmri_data


# Silence third-party chatter: gymnasium routes deprecation notices through its
# own logger, so the warnings filter alone does not catch them.
warnings.filterwarnings('ignore')
gymnasium.logger.min_level = gymnasium.logger.ERROR

MODEL_PARAMS = 'model_params_202606d'
ROWS = (0, 1, 2)          # Vanilla, Masked, bioRNN
N_PC = 5                  # components retained per regime
N_TRIALS = 100            # task trials in the test battery
N_FMRI_SUBJ = 100
N_NULL_DRAWS = 1000       # random subspaces for the chance baseline
SAVE_FIGURES = True

paths = get_paths(MODEL_PARAMS, require='all')   # 'all' -> fMRI is required here
figdir = ensure_dir(paths.figure_dir)

utils.set_font_size(11)
plt.rcParams['svg.fonttype'] = 'none'
sns.set_style('white')
COLORS = utils.get_my_colors(cat_trio=True, as_list=True)


def save(fig, name):
    if SAVE_FIGURES:
        fig.savefig(os.path.join(figdir, name), dpi=300,
                    bbox_inches='tight', pad_inches=0.01)


print(f'models : {paths.model_dir}')
print(f'fMRI   : {paths.fmri_dir}')
print(f'figures: {figdir}')

## Inputs

The evaluation epoch is **derived**, not hard-coded: it is the point at which the slowest
RNN class has reached stable performance, computed from the learning curves in the
performance notebook.

In [ ]:
# Empirical fMRI. Subject selection is seeded, so every notebook analyses the
# same subjects in the same order.
fmri_task, fmri_rest, rest_nsteps, subjects = load_fmri_data(
    paths.data_dir, paths.fmri_dir, N_FMRI_SUBJ, hidden_size=100)

# Models, and the epoch at which every class has converged.
models = dyn.load_model_table(MODEL_PARAMS, rows=ROWS)
performance = perf.load_performance(MODEL_PARAMS, rows=ROWS)
criteria = [perf.criterion_epoch(perf.fit_runs(d['epochs'], d['accuracy'])['t_conv'])
            for d in performance]
ANALYSIS_EPOCH = perf.derive_analysis_epoch(criteria)

print(f'\nanalysis epoch: {ANALYSIS_EPOCH:,} (derived from the learning curves)')
print(f'classes: {", ".join(models.class_label)}')

In [ ]:
# Drive every run under both regimes and summarise with PCA. ~15 s per class.
results = {}
for i in range(len(models)):
    row = models.iloc[i]
    results[row.class_label] = dyn.evaluate_model(
        row, epoch=ANALYSIS_EPOCH, fmri_task_ts=fmri_task, fmri_rest_ts=fmri_rest,
        model_dir=paths.model_dir, n_pc=N_PC, n_trials=N_TRIALS,
        progress=lambda it, lab=row.class_label: tqdm(it, desc=lab, leave=False))

labels = list(results)
# Runs that never learned carry no interpretable dynamics; exclude them.
keep = {lab: dyn.learned_runs(r['accuracy']) for lab, r in results.items()}
for lab in labels:
    print(f'{lab:<12} {keep[lab].sum():3d}/{keep[lab].size} runs learned  '
          f'(mean accuracy {results[lab]["accuracy"][keep[lab]].mean()*100:.1f}%)')

## Fig. S3 — how much RNN variance the five PCs capture

Justifies retaining five components: the elbow of the variance curve falls at five for
every class.

In [ ]:
N_PC_SHOW = 10
fig, axes = plt.subplots(1, len(labels), figsize=(3.4 * len(labels), 3), sharey=True)

for ax, lab, color in zip(np.atleast_1d(axes), labels, COLORS):
    # Refit with more components purely to show where the curve flattens.
    row = models[models.class_label == lab].iloc[0]
    curves = []
    for run in np.flatnonzero(keep[lab])[:20]:      # 20 runs is plenty for the curve
        res = dyn.evaluate_run(row, run, ANALYSIS_EPOCH, paths.model_dir,
                               n_trials=N_TRIALS, rest_nsteps=rest_nsteps,
                               n_pc=N_PC_SHOW)
        curves.append(res['ve_task']['each'] * 100)
    curves = np.vstack(curves)
    x = np.arange(1, N_PC_SHOW + 1)
    ax.errorbar(x, curves.mean(0), yerr=curves.std(0, ddof=1), color=color,
                marker='o', ms=4, lw=1.5, capsize=2)
    ax.axvline(N_PC, color='0.5', ls='--', lw=1)
    total = curves[:, :N_PC].sum(1)
    ax.set_title(f'{lab}\nfirst {N_PC} PCs: {total.mean():.1f} ± {total.std(ddof=1):.1f}%',
                 fontsize=10)
    ax.set_xlabel('Principal component')
    ax.set_xticks(x)

np.atleast_1d(axes)[0].set_ylabel('Variance explained (%)')
sns.despine(fig=fig, right=True, top=True)
fig.tight_layout()
save(fig, 'figS3_pca_variance_explained.svg')
plt.show()

## Fig. 2b — PC loadings on the cortical surface

PC loadings averaged across runs, with signs aligned first: PCA fixes each component only
up to sign, so averaging raw loadings across independently fitted runs would cancel real
structure.

Spatial smoothness is quantified by Moran's I — near zero means no spatial organisation.

In [ ]:
centroids = pd.read_csv(
    os.path.join(paths.data_dir, 'schaefer200_centroids.csv'))[:100]
W_spatial = spatial_null.spatial_weights(centroids[['R', 'A', 'S']].to_numpy())

group_pcs = {lab: dyn.group_components(results[lab]['components_task'][keep[lab]], N_PC)
             for lab in labels}

moran = pd.DataFrame(
    {lab: [spatial_null.morans_i(group_pcs[lab][k], W_spatial) for k in range(N_PC)]
     for lab in labels},
    index=[f'PC{k+1}' for k in range(N_PC)])
print("Moran's I of the group PC maps\n")
print(moran.round(3).to_string())
print('\nmean across PCs:')
for lab in labels:
    print(f'  {lab:<12} {moran[lab].mean():+.3f}')

In [ ]:
# plot_surface returns its own Figure, so each map is rendered to an image and
# composed into a grid here.
def render(fig):
    fig.canvas.draw()
    image = np.asarray(fig.canvas.buffer_rgba())
    plt.close(fig)
    return image


fig, axes = plt.subplots(len(labels), N_PC,
                         figsize=(2.0 * N_PC, 1.5 * len(labels)))

for i, lab in enumerate(labels):
    # One symmetric colour range per class, so PCs are comparable within a row.
    lim = np.abs(group_pcs[lab]).max()
    for k in range(N_PC):
        axes[i, k].imshow(render(plotting.plot_surface(
            group_pcs[lab][k], hemi='lh', n_parcels=200, cmap='coolwarm',
            cblim=(-lim, lim), figsize=(2.0, 1.5))))
        axes[i, k].axis('off')
        if i == 0:
            axes[i, k].set_title(f'PC{k+1}', fontsize=10)
    axes[i, 0].text(-0.05, 0.5, lab, transform=axes[i, 0].transAxes,
                    rotation=90, va='center', ha='right', fontsize=9)

fig.tight_layout()
save(fig, 'fig2b_pc_loadings_surface.svg')
plt.show()

## Fig. 2c — fMRI variance explained, against a random-subspace baseline

Any five-dimensional subspace captures a non-trivial share of variance in
high-dimensional fMRI simply by virtue of its dimensionality, so raw variance explained
cannot be read on its own. Each RNN's value is standardised against a null of
`N_NULL_DRAWS` random five-dimensional subspaces: **z ≈ 0 is chance**.

The same null and the same `ve_to_z` are used by the trajectory and geometry-null
analyses, so the z-scores in different figures are on one scale by construction.

In [ ]:
null = nu.random_subspace_null(
    {'task': fmri_task, 'rest': fmri_rest},
    n_pc=N_PC, n_draws=N_NULL_DRAWS, paired=False, seed=0)

for mod in ('task', 'rest'):
    draws = null[mod]['ve']
    print(f'{mod:<5} null VE: {draws.mean():.4f} ± {draws.std(ddof=1):.4f}  '
          f'(n={draws.size}; expected mean ≈ n_pc/n_nodes = {N_PC/100:.2f})')

z_scores = {lab: {mod: nu.ve_to_z(results[lab][f've_{mod}'][keep[lab]], null, mod)
                  for mod in ('task', 'rest')} for lab in labels}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), sharey=True)
rng = np.random.default_rng(0)

for ax, mod, title in zip(axes, ('task', 'rest'),
                          ('Task fMRI (N-back)', 'Resting-state fMRI')):
    ax.axhspan(-1.96, 1.96, color='0.85', zorder=0)   # chance band
    ax.axhline(0, color='0.5', lw=1, zorder=1)
    for i, (lab, color) in enumerate(zip(labels, COLORS)):
        vals = z_scores[lab][mod]
        body = ax.violinplot(vals, positions=[i], showextrema=False)
        for b in body['bodies']:
            b.set_facecolor(color)
            b.set_alpha(0.4)
        ax.scatter(i + rng.uniform(-0.08, 0.08, vals.size), vals,
                   s=10, color=color, alpha=0.6, linewidths=0)
        ax.hlines(np.median(vals), i - 0.22, i + 0.22, color='k', lw=2, zorder=3)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=15, ha='right')
    ax.set_title(title, fontsize=10)

axes[0].set_ylabel('fMRI variance explained\n($z$ vs random subspaces)')
sns.despine(fig=fig, right=True, top=True)
fig.tight_layout()
save(fig, 'fig2c_fmri_variance_explained_z.svg')
plt.show()

## Fig. 2d — geometry null

Does the correspondence depend on the *veridical* arrangement of regions, or merely on
the spatial autocorrelation that any distance-based kernel imposes?

Two hundred bioRNNs were retrained under **spin-permuted** geometries: the embedding is
rotated on the spherical surface, preserving spatial autocorrelation while destroying the
correspondence to cortex. Each permuted geometry is trained with the same five run
indices as the veridical model, so any difference is attributable to geometry alone.

Because runs are nested within geometries, the test is a variance-components model
separating between-geometry from within-geometry (training) variance.

In [ ]:
import torch

NULL_ROW = 3   # params-CSV row holding the permuted-geometry ensembles

null_models = dyn.load_model_table(MODEL_PARAMS, rows=(NULL_ROW,)).iloc[0]
real_models = dyn.load_model_table(MODEL_PARAMS, rows=(2,)).iloc[0]   # bioRNN
run_ids = [int(x) for x in str(null_models.null_run_ids).split()]

geometry_null = nu.compute_null_distribution(
    null_models, fmri_task, fmri_rest, n_pc=N_PC, modeldir=paths.model_dir,
    device=torch.device('cpu'), epoch=-1, n_trials=N_TRIALS,
    rest_nsteps=rest_nsteps, seed=dyn.EVAL_SEED, n_jobs=-1)

# The veridical bioRNN, evaluated at the same run indices.
veridical = {'task': [], 'rest': []}
for r in run_ids:
    res = dyn.evaluate_run(real_models, r, ANALYSIS_EPOCH, paths.model_dir,
                           n_trials=N_TRIALS, rest_nsteps=rest_nsteps, n_pc=N_PC)
    ve = dyn.fmri_variance_explained(res, fmri_task, fmri_rest)
    veridical['task'].append(ve['task'])
    veridical['rest'].append(ve['rest'])

print(f"{geometry_null['n_perms']} permuted geometries x {len(run_ids)} runs "
      f"(run ids {run_ids})")

In [ ]:
mixed = nu.mixed_null_test(geometry_null['task_runs'],
                           np.asarray(veridical['task']), seed=0)

# Express both the permuted geometries and the veridical model on the same
# z scale used in Fig. 2c, so the panels are directly comparable.
null_z = nu.ve_to_z(np.asarray(geometry_null['task']), null, 'task')
real_z = nu.ve_to_z(np.mean(veridical['task']), null, 'task')

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.hist(null_z, bins=30, color='0.75', edgecolor='none',
        label=f'permuted geometries (n={null_z.size})')
ax.axvline(real_z, color='crimson', lw=2,
           label=f'veridical geometry\n$z_{{geom}}$ = {mixed["z_outlier"]:.2f}, '
                 f'p = {mixed["p_outlier"]:.1e}')
ax.set_xlabel('fMRI variance explained ($z$ vs random subspaces)')
ax.set_ylabel('Number of geometries')
ax.legend(frameon=False, fontsize=9, loc='upper left')
sns.despine(fig=fig, right=True, top=True)
save(fig, 'fig2d_geometry_null.svg')
plt.show()

print(f"between-geometry SD  {mixed['sigma_geom']:.4f}")
print(f"within-geometry SD   {mixed['sigma_train']:.4f}   (ICC = {mixed['icc']:.3f})")
print(f"outlier test         z = {mixed['z_outlier']:.3f}, p = {mixed['p_outlier']:.2e}")
print(f"Wald test            z = {mixed['z_wald']:.3f}, p = {mixed['p_wald']:.2e}")
print(f"normality of permuted means (Shapiro-Wilk) p = {mixed['shapiro_geom_p']:.3f}")
print(f"permuted geometries below veridical: "
      f"{int((np.asarray(geometry_null['task']) < np.mean(veridical['task'])).sum())}"
      f"/{geometry_null['n_perms']}")

## Fig. 2e — correspondence with the cortical hierarchy

Regress the sensorimotor–association (S–A) axis on each class's group PC loadings.
The axis is never shown to the networks, so any correspondence is emergent.

The empirical fMRI PCs provide the benchmark: they are built the same way, by
aligning and averaging per-subject PCA loadings. Because fMRI is one of the features
used to construct the S–A axis in the first place, this bar is a ceiling rather than
a competitor.

Values are **adjusted** R², which accounts for the fit gained from five predictors
over ~100 parcels.

In [ ]:
sa_axis = np.load(os.path.join(paths.data_dir, 'schaefer200_sa-axis.npy'))[:100]

# Empirical benchmark: per-subject fMRI PCA, aligned and averaged the same way
# as the RNN runs. Component order varies far more across subjects than across
# runs, so the permutation matching inside align_and_average_components matters
# a great deal here.
fmri_pcas = [pca_utils.fit_pca(fmri_task[:, :, s], N_PC)
             for s in range(fmri_task.shape[2])]
fmri_components, align_diag = pca_utils.align_and_average_components(
    fmri_pcas, N_PC, return_diagnostics=True)

bars = {lab: spatial_null.map_r2(group_pcs[lab], sa_axis, k=N_PC) for lab in labels}
bars['Task fMRI'] = spatial_null.map_r2(fmri_components, sa_axis, k=N_PC)
bars = {k: spatial_null.map_r2_adjusted(v, n=sa_axis.size, k=N_PC)
        for k, v in bars.items()}

chance_r2 = spatial_null.map_r2_adjusted(
    spatial_null.map_r2_critical(0.05, n=sa_axis.size, k=N_PC), n=sa_axis.size, k=N_PC)

print('per-PC match quality across subjects (mean cosine): '
      f"{np.round(align_diag['mean_cosine'], 2)}")

names = list(bars)
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.bar(range(len(names)), [bars[n] for n in names],
       color=list(COLORS) + ['0.45'], alpha=0.85, width=0.6)
ax.axhline(chance_r2, color='0.4', ls='--', lw=1)
ax.text(len(names) - 0.5, chance_r2, ' p = 0.05', color='0.4',
        va='bottom', ha='right', fontsize=9)
for i, n in enumerate(names):
    ax.text(i, bars[n] + 0.015, f'{bars[n]:.2f}', ha='center', fontsize=10)

ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('S–A axis variance explained (adj. $R^2$)')
ax.set_ylim(0, max(bars.values()) * 1.25)
sns.despine(fig=fig, right=True, top=True)
save(fig, 'fig2e_sa_axis_variance_explained.svg')
plt.show()

## Fig. S4 — subject-specific correspondence

Group-level agreement could hold without any individual specificity. Here each bioRNN is
compared to **each subject's own** geometry, and asked how well it predicts **that
subject's** fMRI.

One correlation per subject, taken across the bioRNN ensemble: a positive value means the
networks whose connectivity most resembles a given brain's geometry are the ones that best
predict that brain's activity.

In [ ]:
import scipy.stats as sp_stats

coords_file = os.path.join(
    paths.fmri_dir, 'HCP_YA_Coords_Schaefer2018_200Parcels_7Networks.npz')
subject_kernels = dyn.subject_geometry_kernels(coords_file, subjects, hidden_size=100)

bio = results['bioRNN']
# Every run contributes here: the question is how weight-geometry
# similarity tracks prediction across the ensemble, which a run that
# failed to learn still speaks to.
bio_runs = bio['runs']

similarity = np.zeros((bio_runs.size, len(subjects)))
ve_subject = np.zeros((bio_runs.size, len(subjects)))
for i, r in enumerate(tqdm(bio_runs, desc='bioRNN runs', leave=False)):
    weights = dyn.load_hidden_weights(real_models, r, ANALYSIS_EPOCH, paths.model_dir)
    res = dyn.evaluate_run(real_models, r, ANALYSIS_EPOCH, paths.model_dir,
                           n_trials=N_TRIALS, rest_nsteps=rest_nsteps, n_pc=N_PC)
    ve_subject[i] = pca_utils.subspace_ve_per_subject(fmri_task, res['pca_task'])
    for s in range(len(subjects)):
        similarity[i, s] = dyn.weight_kernel_similarity(
            weights, subject_kernels[s], center=True)

# One Spearman correlation per subject, across the ensemble.
rho = np.array([sp_stats.spearmanr(similarity[:, s], ve_subject[:, s]).statistic
                for s in range(len(subjects))])
t_stat, p_val = sp_stats.ttest_1samp(rho, 0)
print(f'rho = {rho.mean():.3f} ± {rho.std(ddof=1):.3f} across {rho.size} subjects '
      f'(t = {t_stat:.1f}, p = {p_val:.1e})')

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 3.8))
rng = np.random.default_rng(0)

body = ax.violinplot(rho, positions=[0], showextrema=False)
for b in body['bodies']:
    b.set_facecolor(COLORS[2])
    b.set_alpha(0.4)
ax.scatter(rng.uniform(-0.08, 0.08, rho.size), rho, s=12,
           color=COLORS[2], alpha=0.6, linewidths=0)
ax.hlines(np.median(rho), -0.22, 0.22, color='k', lw=2, zorder=3)
ax.axhline(0, color='0.5', ls=':', lw=1)

ax.set_xticks([0])
ax.set_xticklabels(['bioRNN'])
ax.set_ylabel('Weight–geometry similarity vs\nfMRI prediction ($r_S$, per subject)')
ax.set_title(f'$r_S$ = {rho.mean():.2f} ± {rho.std(ddof=1):.2f}', fontsize=10)
sns.despine(fig=fig, right=True, top=True)
save(fig, 'figS4_subject_specific_effect.svg')
plt.show()

## Reported values

In [ ]:
header = (f'{"class":<13}{"runs":>6}{"PCA VE (%)":>16}{"Moran I":>10}'
          f'{"z (task)":>14}{"z (rest)":>14}{"adj R2":>9}')
print(header + '\n' + '-' * len(header))
for lab in labels:
    r, m = results[lab], keep[lab]
    pcv = r['explained_variance_task'][m].sum(axis=1) * 100
    zt, zr = z_scores[lab]['task'], z_scores[lab]['rest']
    print(f'{lab:<13}{m.sum():>6}'
          f'{pcv.mean():>9.1f} ± {pcv.std(ddof=1):<4.1f}'
          f'{moran[lab].mean():>+10.3f}'
          f'{zt.mean():>8.2f} ± {zt.std(ddof=1):<4.2f}'
          f'{zr.mean():>8.2f} ± {zr.std(ddof=1):<4.2f}'
          f'{bars[lab]:>9.3f}')

print(f"\nEmpirical benchmark (task fMRI): adj R2 = {bars['Task fMRI']:.3f}")
print(f'Geometry null: z_geom = {mixed["z_outlier"]:.2f}, p = {mixed["p_outlier"]:.1e} '
      f'({geometry_null["n_perms"]} permuted geometries)')
print(f'Subject-specific effect: rho = {rho.mean():.3f} ± {rho.std(ddof=1):.3f}')
print(f'\nEvaluated at epoch {ANALYSIS_EPOCH:,}; {N_FMRI_SUBJ} subjects; '
      f'null = {N_NULL_DRAWS} random {N_PC}-D subspaces (seed 0).')